**Basic 스태킹 모델**

데이터 로딩

In [1]:
import numpy as np

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

cancer_data = load_breast_cancer()

X_data = cancer_data.data
y_label = cancer_data.target

X_train, X_test, y_train, y_test = train_test_split(X_data, y_label, test_size=0.2, random_state=0)


**개별 Classifier와 최종 Stacking 데이터를 학습할 메타 Classifier 생성**

In [5]:
# 개별 ML 모델을 위한 Classifier 생성

knn_clf = KNeighborsClassifier(n_neighbors=4)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=0)
dt_clf = DecisionTreeClassifier()
ada_clf = AdaBoostClassifier(n_estimators=100)

# 최종 Stacking 모델을 위한 Classifier 생성
lr_final = LogisticRegression(C=10)

**개별 Classifier 학습/예측/평가**

In [6]:
# 개별 모델들 학습
knn_clf.fit(X_train, y_train)
rf_clf.fit(X_train, y_train)
dt_clf.fit(X_train, y_train)
ada_clf.fit(X_train, y_train)

AdaBoostClassifier(n_estimators=100)

In [7]:
knn_pred = knn_clf.predict(X_test)
rf_pred = rf_clf.predict(X_test)
dt_pred = dt_clf.predict(X_test)
ada_pred = ada_clf.predict(X_test)

# 각 개별 분류기의 정확도 계산 및 출력
knn_accuracy = accuracy_score(y_test, knn_pred)
rf_accuracy = accuracy_score(y_test, rf_pred)
dt_accuracy = accuracy_score(y_test, dt_pred)
ada_accuracy = accuracy_score(y_test, ada_pred)

print(f"KNN 분류기 정확도: {knn_accuracy:.4f}")
print(f"Random Forest 분류기 정확도: {rf_accuracy:.4f}")
print(f"Decision Tree 분류기 정확도: {dt_accuracy:.4f}")
print(f"AdaBoost 분류기 정확도: {ada_accuracy:.4f}")

KNN 분류기 정확도: 0.9211
Random Forest 분류기 정확도: 0.9649
Decision Tree 분류기 정확도: 0.9123
AdaBoost 분류기 정확도: 0.9737


**개별 모델의 예측 결과를 메타 모델이 학습할 수 있도록 스태킹 형태로 재 생성**

In [8]:
pred = np.array([knn_pred, rf_pred, dt_pred, ada_pred])
print(pred.shape) # 4행 114열
pred = np.transpose(pred) # 전치 행렬로 열과 행을 교환합니다.
print(pred.shape) # 114행 4열

(4, 114)
(114, 4)


**메타 모델 학습/예측/평가**

In [10]:
lr_final.fit(pred, y_test)
final = lr_final.predict(pred)

print('최종 메타 모델의 예측 정확도 {0: .4f}'.format(accuracy_score(y_test, final))) # 학습데이터에 대해서 overfitting

최종 메타 모델의 예측 정확도  0.9825


In [17]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# 개별 기반 모델에서 최종 메타 모델이 사용할 학습 및 테스트용 데이터를 생성하기 위한 함수
def get_stacking_base_datasets(model, X_train_n, y_train_n, X_test_n, n_folds):
    # 지정된 n_folds 값으로 KFold 생성
    kf = KFold(n_splits=n_folds, shuffle=False)
    
    # 추후에 메타 모델이 사용할 학습 데이터 반환을 위한 넘파이 배열 초기화
    train_fold_pred = np.zeros((X_train_n.shape[0], 1)) # 2차원으로 바꿉니다. ex) 400건의 데이터가 오면 (400, 1)
    test_pred = np.zeros((X_test_n.shape[0], n_folds)) 
    print(model.__class__.__name__, ' 모델 시작')
    
    for folder_counter, (train_index, valid_index) in enumerate(kf.split(X_train_n)):
        # 입력된 학습 데이터에서 기반 모델이 학습/예측할 폴드 데이터 셋 추출
        print("\t 폴드 세트: ", folder_counter, ' 시작')
        X_tr = X_train_n[train_index]
        y_tr = y_train_n[train_index]
        X_te = X_train_n[valid_index]
        
        # 폴드 세트 내부에서 다시 만들어진 학습 데이터로 기반 모델의 학습 수행
        model.fit(X_tr, y_tr)
        
        # 폴드 세트 내부에서 다시 만들어진 검증 데이터로 기반 모델 예측 후 데이터 저장
        train_fold_pred[valid_index, :] = model.predict(X_te).reshape(-1, 1) # 2차원으로 reshape
        
        # 입력된 원본 테스트 데이터를 폴드 세트 내 학습된 기반 모델에서 예측후 데이터 저장
        test_pred[:, folder_counter] = model.predict(X_test_n)
    
    # 폴드 세트 내에서 원본 테스트 데이터를 예측한 데이터를 평균하여 테스트 데이터로 생성
    test_pred_mean = np.mean(test_pred, axis=1).reshape(-1, 1)
    
    # train_fold_pred는 최종 메타 모델이 사용하는 학습 데이터, test_pred_mean은 테스트 데이터
    return train_fold_pred, test_pred_mean
        

In [18]:
knn_train, knn_test = get_stacking_base_datasets(knn_clf, X_train, y_train, X_test, n_folds=7)
rf_train, rf_test = get_stacking_base_datasets(rf_clf, X_train, y_train, X_test, n_folds=7)
dt_train, dt_test = get_stacking_base_datasets(dt_clf, X_train, y_train, X_test, n_folds=7)
ada_train, ada_test = get_stacking_base_datasets(ada_clf, X_train, y_train, X_test, n_folds=7)


KNeighborsClassifier  모델 시작
	 폴드 세트:  0  시작
	 폴드 세트:  1  시작
	 폴드 세트:  2  시작
	 폴드 세트:  3  시작
	 폴드 세트:  4  시작
	 폴드 세트:  5  시작
	 폴드 세트:  6  시작
RandomForestClassifier  모델 시작
	 폴드 세트:  0  시작
	 폴드 세트:  1  시작
	 폴드 세트:  2  시작
	 폴드 세트:  3  시작
	 폴드 세트:  4  시작
	 폴드 세트:  5  시작
	 폴드 세트:  6  시작
DecisionTreeClassifier  모델 시작
	 폴드 세트:  0  시작
	 폴드 세트:  1  시작
	 폴드 세트:  2  시작
	 폴드 세트:  3  시작
	 폴드 세트:  4  시작
	 폴드 세트:  5  시작
	 폴드 세트:  6  시작
AdaBoostClassifier  모델 시작
	 폴드 세트:  0  시작
	 폴드 세트:  1  시작
	 폴드 세트:  2  시작
	 폴드 세트:  3  시작
	 폴드 세트:  4  시작
	 폴드 세트:  5  시작
	 폴드 세트:  6  시작


In [19]:
Stack_final_X_train = np.concatenate((knn_train, rf_train, dt_train, ada_train), axis=1)
Stack_final_X_test = np.concatenate((knn_test, rf_test, dt_test, ada_test), axis=1)

print(X_train.shape, X_test.shape)
print(Stack_final_X_train.shape, Stack_final_X_test.shape)

(455, 30) (114, 30)
(455, 4) (114, 4)


In [20]:
lr_final.fit(Stack_final_X_train, y_train)
stack_final = lr_final.predict(Stack_final_X_test)

print('최종 메타 모델의 예측 정확도 : {0: .4f}'.format(accuracy_score(y_test, stack_final)))

최종 메타 모델의 예측 정확도 :  0.9649


/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/songbeom/PythonWorkSpace/machineLearning/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
